# Gold — Fatos



In [ ]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

In [ ]:
load_dotenv()

is_docker = os.path.exists("/.dockerenv")
MINIO_HOST = "minio:9000" if is_docker else "localhost:9000"
MINIO_USER = os.getenv("MINIO_ROOT_USER",     "minioadmin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

builder = (
    SparkSession.builder
    .appName("gold-fatos")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.defaultFS", "file:///")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

cliente_minio = Minio(
    MINIO_HOST, access_key=MINIO_USER, secret_key=MINIO_PASS, secure=False
)

In [ ]:
buckets = {b.name for b in cliente_minio.list_buckets()}
print("Buckets:", buckets)
if "gold" not in buckets:
    cliente_minio.make_bucket("gold")
    print("Bucket 'gold' criado.")

In [ ]:
def download_delta_do_minio(bucket: str, prefix: str,
                            cliente: Minio, local_dir: Path) -> None:
    """Baixa toda a estrutura Delta (parquet + _delta_log) do MinIO para local."""
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True)
    objetos = list(cliente.list_objects(bucket, prefix=f"{prefix}/", recursive=True))
    print(f"  {len(objetos)} arquivo(s) em {bucket}/{prefix}/")
    for obj in objetos:
        relative = obj.object_name[len(prefix) + 1:]
        destino = local_dir / relative
        destino.parent.mkdir(parents=True, exist_ok=True)
        cliente.fget_object(bucket, obj.object_name, str(destino))


def upload_delta_para_minio(local_dir: Path, bucket: str, prefix: str,
                            cliente: Minio) -> None:
    """Sobe recursivamente a estrutura Delta para o MinIO."""
    for arq in sorted(local_dir.rglob("*")):
        if arq.is_file():
            object_name = f"{prefix}/{arq.relative_to(local_dir)}"
            cliente.fput_object(bucket, object_name, str(arq))
            print(f"  → {object_name}")

## fato_reproducao



In [ ]:
print("\n" + "=" * 55)
print("Gold: fato_reproducao")
print("=" * 55)

local_repro = Path("/tmp/silver_local/reproducoes")
download_delta_do_minio("silver", "fatos/reproducoes", cliente_minio, local_repro)
df_repro = spark.read.format("delta").load(str(local_repro))
print(f"Silver lido: {df_repro.count():,} plays válidos")

local_musicas = Path("/tmp/silver_local/musicas")
download_delta_do_minio("silver", "dimensoes/musicas", cliente_minio, local_musicas)
df_musica_artista = (
    spark.read.format("delta").load(str(local_musicas))
    .select(F.col("id").alias("musica_id"), "artista_id")
)

df_fato_repro = (
    df_repro
    .join(df_musica_artista, on="musica_id", how="left")
    .withColumn("data_id", F.date_format("timestamp", "yyyyMMdd").cast("int"))
    .withColumn("contagem", F.lit(1))
    .select(
        "data_id", "usuario_id", "musica_id", "artista_id",
        "ms_tocados", "completou", "contagem", "ano_mes",
    )
)

local_gold = Path("/tmp/gold/fato_reproducao")
if local_gold.exists():
    shutil.rmtree(local_gold)

df_fato_repro.write.format("delta").mode("overwrite").partitionBy("ano_mes").save(str(local_gold))
print(f"Gold Delta escrito em {local_gold}")

upload_delta_para_minio(local_gold, "gold", "fatos/fato_reproducao", cliente_minio)
print(f"fato_reproducao concluído — {df_fato_repro.count():,} registros.")

## fato_pagamento



In [ ]:
print("\n" + "=" * 55)
print("Gold: fato_pagamento")
print("=" * 55)

local_pag = Path("/tmp/silver_local/pagamentos")
download_delta_do_minio("silver", "fatos/pagamentos", cliente_minio, local_pag)
df_pag = spark.read.format("delta").load(str(local_pag))
print(f"Silver lido: {df_pag.count():,} pagamentos")

local_assin = Path("/tmp/silver_local/assinaturas")
download_delta_do_minio("silver", "fatos/assinaturas", cliente_minio, local_assin)
df_assin = (
    spark.read.format("delta").load(str(local_assin))
    .select(F.col("id").alias("assinatura_id"), "usuario_id", "plano_id")
)

df_fato_pag = (
    df_pag
    .join(df_assin, on="assinatura_id", how="left")
    .withColumn("data_id", F.date_format(F.col("data").cast("timestamp"), "yyyyMMdd").cast("int"))
    .withColumn("pago", F.col("status") == F.lit("pago"))
    .select("data_id", "usuario_id", "plano_id", "valor", "pago", "ano_mes")
)

local_gold = Path("/tmp/gold/fato_pagamento")
if local_gold.exists():
    shutil.rmtree(local_gold)

df_fato_pag.write.format("delta").mode("overwrite").partitionBy("ano_mes").save(str(local_gold))
print(f"Gold Delta escrito em {local_gold}")

upload_delta_para_minio(local_gold, "gold", "fatos/fato_pagamento", cliente_minio)
print(f"fato_pagamento concluído — {df_fato_pag.count():,} registros.")
print("\nTransformação Gold de todos os fatos concluída.")

In [ ]:
spark.stop()